In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.2 RMSNorm

LayerNorm centers (subtract mean) then scales (divide by std). RMSNorm skips the
centering and just divides by the root-mean-square. Fewer ops, no bias, and in
practice no loss of quality, which is why modern models use it.

In [ ]:
from pocketlm import RMSNorm
from torch import nn

x = torch.randn(2, 4, 8) * 5 + 3
rms = RMSNorm(8)(x)
ln = nn.LayerNorm(8)(x)
print("RMSNorm output RMS ~1:", round(rms.pow(2).mean(-1).sqrt().mean().item(), 4))
print("RMSNorm params:", sum(p.numel() for p in RMSNorm(8).parameters()),
      " LayerNorm params:", sum(p.numel() for p in nn.LayerNorm(8).parameters()))

RMSNorm has only the scale weight (8 params here); LayerNorm also has a bias (16
total). The saving is small per layer but adds up, and the simpler op is faster.

In [ ]:
rms_out = RMSNorm(8)(x)
assert abs(rms_out.pow(2).mean(-1).sqrt().mean().item() - 1.0) < 0.05
assert sum(p.numel() for p in RMSNorm(8).parameters()) < \
       sum(p.numel() for p in nn.LayerNorm(8).parameters())